# TradeSystem 性能分析

先确认文件来自哪次运行，再看延迟。这里分开分析诊断日志、组件 CSV 和逐请求 CSV，不混合不同进程、轮次或版本，也不删除慢样本。

- 当前脚本为每次运行建立独立目录；数据入口改为 `TRADE_ANALYSIS_DIR`，不再扫描仓库根目录的旧日志。
- 默认 Release 的 `TRADE_PERF_TRACE_EVERY=0`，没有 RDTSC/TTT 记录是正常现象，不等于零延迟。
- TSC 没有标定时只显示 ticks；CPU 标称 GHz 不能代替 TSC 频率。
- TTT 没有请求 ID，且取自 system_clock；只能查看记录覆盖，不能把相邻行相减当作一笔订单的耗时。

使用方法见 [notebooks/README.md](README.md)，改动与验证见 [09 报告](../record/09_notebook_analysis.md)。


In [ ]:
import html
import importlib.util
import os
from pathlib import Path
from IPython.display import HTML, display

# 兼容在仓库根目录或 notebooks 目录打开；不写死机器路径。
here = Path(os.environ.get("TRADE_REPO_DIR", Path.cwd())).resolve()
helper = next((candidate for base in [here, *here.parents]
               for candidate in [base / "perf_analysis.py", base / "notebooks/perf_analysis.py"]
               if candidate.is_file()), None)
if helper is None:
    raise FileNotFoundError("未找到 perf_analysis.py；请从仓库打开或设置 TRADE_REPO_DIR。")
spec = importlib.util.spec_from_file_location("trade_notebook_analysis", helper)
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)

def show_table(rows):
    if not rows:
        print("没有这一类记录。")
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    headings = "".join(f"<th>{html.escape(str(key))}</th>" for key in columns)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(str(row.get(key, '')))}</td>"
                                  for key in columns) + "</tr>" for row in rows)
    display(HTML(f'<div style="overflow-x:auto"><table><thead><tr>{headings}</tr></thead>'
                 f'<tbody>{body}</tbody></table></div>'))


## 选择输入

把 `DATA_DIR` 改成某次联合运行目录、组件结果目录或逐请求测量目录，也可在启动 Jupyter 前设置 `TRADE_ANALYSIS_DIR`。不要把整个历史归档当成一个实验。下表保留文件路径，每个文件分别统计，不合并轮次。

`TSC_HZ` 默认不填。只有明确完成 TSC 标定时才填实测 Hz；它仅用于选定实验中的 RDTSC 换算。逐请求文件通常使用同目录 `result.json` 的单调时钟窗口；只有缺少元数据时才手动填写 `REQUEST_WINDOW_NS=(start_ns, end_ns)`。


In [ ]:
DATA_DIR = os.environ.get("TRADE_ANALYSIS_DIR") or None
TSC_HZ = None
REQUEST_WINDOW_NS = None
PLOT_SOURCE = None  # 可填表格中的 source；默认画第一个非 clock_empty 序列。

data = analysis.analyze_directory(DATA_DIR, TSC_HZ, REQUEST_WINDOW_NS)
print("数据目录：", DATA_DIR or "尚未指定")
show_table([{"类别": key, "文件/标签数": len(data[key])}
            for key in ["rdtsc", "ttt", "components", "requests"]])
for issue in data["issues"]:
    print("提示/数据问题：", issue)


## 诊断计时

RDTSC 表保留全部非负记录，包括相同时间、标签、数值的重复行。每个来源文件、标签各自统计；有解析问题时，上面的提示不能忽略。频率未标定时，表中的均值和分位数单位为 **TSC ticks**，不是纳秒或 CPU 核心周期。

TTT 表仅显示条数和原始墙钟时间戳范围，没有请求相关键、单调时钟及完整采样保证，无法可靠推导分段链路延迟。


In [ ]:
print("RDTSC 诊断样本：")
show_table(data["rdtsc"])
print("TTT 覆盖情况（epoch_ns 不是延迟）：")
show_table(data["ttt"])


## 组件样本

支持当前基准入口的 `operation,elapsed_ns` CSV，以及池压力实验的 `cycle,pairs,elapsed_ns` CSV。前者是程序定义的每个样本计时，可能是一对入出队、一次缓冲复制或一个快照周期，包含读钟开销；后者是每周期平均每对分配／释放的成本，不是单对操作的延迟分布。其他列结构会提示未支持，避免误用单位。

分位数复用仓库分析器的 nearest-rank 算法，不裁剪 1%/99% 两端，也保留零值。当前组件输出有 `result.txt` 时，核对原始 CSV 的数量和 P50/P95/P99/max。历史批次探针可能采用不同边界索引；这里的重算不覆盖原报告的原值。

组件成绩不包含完整订单链；`OptMemPool`、`OptLogger` 是当前实现的别名，两组属于重复对照。不能仅凭两组快慢声称源码优化收益。


In [ ]:
show_table(data["components"])


## 逐请求往返

复用 `tests/02_reproducible_measurement/analyze.py`，按请求 ID、状态和时间戳校验 CSV，并和 `result.json` 重算核对。

- RTT 从完整请求帧实际发出到收到完整回报，范围是测量生成器与交易所，不包含客户端策略、网关和行情消费器。
- `completed_rtt_*` 只描述已完成请求。TIMEOUT、UNSENT 单独显示；存在这些项时，所有输入请求的 P99 显示为不可精确估计，不把未完成项当作零。
- `window_completed_per_s` 只统计输入窗口内完成的回报；完成队列排空后的 cohort 数量是另一口径。
- 归档里的 `warmup_events.csv` 单独保留并提示，不纳入正式请求统计。
- `correctness=passed` 来自原业务验收元数据，不是仅凭 CSV 有数值。未提供验收状态或失败时保留相应标记。

旧版相邻 TTT 行的 `diff()` 已移除，这里没有靠时间排序猜测请求配对。


In [ ]:
show_table(data["requests"])


## 保留完整样本的图

横轴是该文件的样本序号；右图是经验累计分布。两图保留全部样本和尾部。不同单位的序列分别画，不能把 TSC ticks、组件纳秒和 RTT 混在一条曲线上。图只描述选中的一个来源，不自动计算旧／新收益。

安装了 matplotlib 才绘图；表格分析无需 pandas、NumPy、Plotly 或 hvplot。


In [ ]:
try:
    import matplotlib.pyplot as plt
    get_ipython().run_line_magic("matplotlib", "inline")
except ImportError:
    plt = None
    print("未安装 matplotlib，表格分析已完成；安装后可绘图。")

series = data["series"]
selected = PLOT_SOURCE or next((key for key, value in series.items()
                               if "clock_empty" not in key and value["values"]), None)
if plt is not None and selected is not None:
    if selected not in series:
        raise ValueError("PLOT_SOURCE 不在当前来源列表中。")
    values, unit = series[selected]["values"], series[selected]["unit"]
    if values:
        ordered = sorted(values)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(range(len(values)), values, linewidth=0.7)
        axes[0].set(xlabel="Sample index", ylabel=unit, title="All samples")
        axes[1].step(ordered, [(i + 1) / len(ordered) for i in range(len(ordered))], where="post")
        axes[1].set(xlabel=unit, ylabel="Cumulative fraction", title="Empirical CDF")
        fig.suptitle(selected)
        fig.tight_layout()
        plt.show()
        plt.close(fig)
else:
    print("当前没有可绘制的计时样本。")
